# Chapter 1 — Tokenizer from scratch
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch01_tokenizer.ipynb)

Based on `oreilly-japan/deep-learning-from-scratch-6/ch01`. This notebook implements character/byte tokenization, BPE training, encode/decode, special tokens, and pre-tokenization. GPU is not required for this chapter.

In [ ]:
from collections import Counter
import re
text = 'low lower lowest new newer newest'
print(text)

## 1. Character and byte tokenizers

In [ ]:
chars = sorted(set(text))
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}
encoded = [stoi[c] for c in text]
decoded = ''.join(itos[i] for i in encoded)
print(encoded[:20])
print(decoded)

byte_ids = list('안녕 LLM'.encode('utf-8'))
print(byte_ids)
print(bytes(byte_ids).decode('utf-8'))

## 2. Byte Pair Encoding (BPE)
Repeatedly merge the most frequent adjacent token pair. The implementation below intentionally keeps the algorithm visible rather than hiding it behind a tokenizer library.

In [ ]:
def pair_counts(ids):
    return Counter(zip(ids, ids[1:]))

def merge_pair(ids, pair, new_id):
    out = []
    i = 0
    while i < len(ids):
        if i + 1 < len(ids) and (ids[i], ids[i+1]) == pair:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

def train_bpe(raw_bytes, vocab_size=280):
    ids = list(raw_bytes)
    merges = {}
    next_id = 256
    while next_id < vocab_size:
        counts = pair_counts(ids)
        if not counts: break
        pair = max(counts, key=counts.get)
        ids = merge_pair(ids, pair, next_id)
        merges[pair] = next_id
        next_id += 1
    return merges

train_text = ('low lower lowest new newer newest ' * 20).encode('utf-8')
merges = train_bpe(train_text, vocab_size=280)
print('number of merges:', len(merges))
print(list(merges.items())[:8])

In [ ]:
def bpe_encode(s, merges):
    ids = list(s.encode('utf-8'))
    while len(ids) >= 2:
        pairs = pair_counts(ids)
        candidates = [(merges[p], p) for p in pairs if p in merges]
        if not candidates: break
        _, pair = min(candidates)
        ids = merge_pair(ids, pair, merges[pair])
    return ids

vocab = {i: bytes([i]) for i in range(256)}
for pair, idx in sorted(merges.items(), key=lambda x:x[1]):
    vocab[idx] = vocab[pair[0]] + vocab[pair[1]]

def bpe_decode(ids):
    return b''.join(vocab[i] for i in ids).decode('utf-8', errors='replace')

sample = 'newest lower'
ids = bpe_encode(sample, merges)
print(ids)
print(bpe_decode(ids))

## 3. Special tokens and pre-tokenization

In [ ]:
SPECIAL = {'<|bos|>': 100000, '<|eos|>': 100001}
pattern = r"\w+|[^\w\s]+"
sentence = 'Hello, tokenizer! 123'
pieces = re.findall(pattern, sentence)
print(pieces)
print(SPECIAL)

## Checkpoint
You should now be able to explain why byte-level BPE guarantees coverage, what a merge table stores, and why pre-tokenization changes the statistics seen by BPE.

Upstream reference: https://github.com/oreilly-japan/deep-learning-from-scratch-6/tree/main/ch01